# Multimodal LLMs — from representations to visual question answering

**Practical session · 60 minutes · Python / PyTorch / Hugging Face**

In this practical, we will use one recurring example from the course: an orange kitten lying on its back. We will move through the three mechanisms introduced in the lecture:

1. **Encoding** — turn image and text inputs into numerical representations.
2. **Alignment** — compare those representations with CLIP.
3. **Fusion** — give an image and a question to a vision-language model and generate an answer.

The notebook is designed around experiments, not implementation. Most code is provided. You will mainly change a few text prompts, inspect outputs and explain what they mean.

> **Learning outcomes**
> - Explain what image and text embeddings represent.
> - Use CLIP for zero-shot classification and text-to-image retrieval.
> - Distinguish similarity-based alignment from open-ended visual question answering.
> - Explain the high-level difference between LLaVA-style projection and Flamingo-style cross-attention.

**Run the notebook from top to bottom.** No training is required.

## Session map

| Time | Activity | Main idea |
|---:|---|---|
| 0–5 min | Introduction and setup | A multimodal input contains more than text |
| 5–12 min | Image and text representations | Encoding |
| 12–27 min | CLIP zero-shot classification | Contrastive alignment |
| 27–37 min | Text-to-image retrieval | Reuse the shared space |
| 37–51 min | Visual question answering | Multimodal fusion |
| 51–56 min | LLaVA vs Flamingo | Two fusion mechanisms |
| 56–60 min | Recap quiz | Encoding vs alignment vs fusion |

### What you need to know

- The environment should already contain **PyTorch**, **Transformers**, **Pillow**, **Matplotlib** and **NumPy**.
- The model weights should be cached locally or available over the network.
- The three sample images are included in the practical bundle. No dataset download is needed for the core exercises.
- A local image folder can optionally be added to the retrieval gallery (see Part 3).

**Only edit the clearly marked prompt cells.** You do not need to implement preprocessing, tokenization, batching or GPU management.

## 0. Setup and first observation (0–5 min)

Start with the same question as in the lecture:

> **What can we observe in this image, and what would be an inference?**

Look carefully before running any model. For example, closed eyes are observable; being asleep is an interpretation that may or may not be correct.

In [ ]:
# Imports and runtime information — no installation should be needed during the session.
import os
import math
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

import transformers
from transformers import CLIPModel, CLIPProcessor

print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"BF16 support: {torch.cuda.is_bf16_supported()}")
else:
    print("WARNING: CUDA is not available. The notebook can run on CPU, but model inference will be slower.")

In [ ]:
# Configuration.
# If your institution has pre-downloaded checkpoints, replace the model IDs below
# with their local directories, or set the matching environment variables before
# starting Jupyter (CLIP_MODEL_ID and VLM_MODEL_ID).

CLIP_MODEL_ID = os.environ.get("CLIP_MODEL_ID", "openai/clip-vit-base-patch32")
VLM_MODEL_ID = os.environ.get("VLM_MODEL_ID", "HuggingFaceTB/SmolVLM2-2.2B-Instruct")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float32
)

ASSET_DIR = Path("assets")
if not ASSET_DIR.exists() and (Path.cwd() / "multimodal_lab_assets" / "assets").exists():
    ASSET_DIR = Path.cwd() / "multimodal_lab_assets" / "assets"

print("Device:", DEVICE)
print("VLM dtype:", MODEL_DTYPE)
print("CLIP checkpoint:", CLIP_MODEL_ID)
print("VLM checkpoint:", VLM_MODEL_ID)
print("Sample assets:", ASSET_DIR.resolve())

### Inspect the sample gallery

The core gallery contains the kitten from the opening slide and two dog photographs used in the course material. They are deliberately small so that the exercises remain fast. If you have a pre-downloaded image dataset, you can add its folder later.

In [ ]:
# Load the images shipped next to this notebook.
def load_rgb(path):
    """Open an image safely and return an RGB PIL image."""
    with Image.open(path) as image:
        return image.convert("RGB")

sample_specs = [
    ("kitten.png", "orange kitten"),
    ("puppy.png", "golden retriever puppy"),
    ("dog_in_flowers.png", "dog in a field of flowers"),
]

gallery = []
for filename, label in sample_specs:
    path = ASSET_DIR / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Extract the full practical ZIP so that the assets/ folder "
            "is next to the notebook."
        )
    gallery.append({"path": path, "label": label, "image": load_rgb(path)})

# Optional: include up to 15 additional images from a local pre-downloaded dataset.
# Set MM_LAB_GALLERY_DIR to a directory containing .jpg/.jpeg/.png/.webp files.
extra_dir = os.environ.get("MM_LAB_GALLERY_DIR", "")
if extra_dir and Path(extra_dir).is_dir():
    supported = {".jpg", ".jpeg", ".png", ".webp"}
    extra_paths = [p for p in sorted(Path(extra_dir).glob("*")) if p.suffix.lower() in supported]
    added = 0
    for path in extra_paths[:15]:
        try:
            gallery.append({"path": path, "label": path.stem.replace("_", " "), "image": load_rgb(path)})
            added += 1
        except Exception as exc:
            print(f"Skipping {path.name}: {exc}")
    print(f"Added {added} image(s) from {extra_dir}")
else:
    print("Using the 3 bundled sample images. (Optional local gallery: set MM_LAB_GALLERY_DIR.)")

fig, axes = plt.subplots(1, len(gallery), figsize=(4 * len(gallery), 3.4))
if len(gallery) == 1:
    axes = [axes]
for ax, item in zip(axes, gallery):
    ax.imshow(item["image"])
    ax.set_title(item["label"])
    ax.axis("off")
plt.tight_layout()
plt.show()

kitten_path = ASSET_DIR / "kitten.png"
kitten_image = load_rgb(kitten_path)

---

## 1. From images and text to vectors (5–12 min)

A common CLIP-style pipeline has two encoders:

- an **image encoder** maps pixels to an image representation;
- a **text encoder** maps tokenized text to a text representation.

The model is trained so that matching image–text pairs are close in a shared space. Equal vector dimensions alone would not guarantee that this comparison is meaningful: the encoders must be trained to align their representations.

### Load CLIP

We use `openai/clip-vit-base-patch32` as a compact, familiar baseline. If a local checkpoint is available, set `CLIP_MODEL_ID` in the configuration cell.

In [ ]:
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(DEVICE).eval()
print("CLIP loaded.")

### Encode the same kitten image and one description

The helper functions below hide preprocessing and device handling. The important operations are the encoder calls and the shapes of their outputs.

In [ ]:
def encode_images(images):
    batch = clip_processor(images=images, return_tensors="pt")
    batch = {key: value.to(DEVICE) for key, value in batch.items()}
    with torch.inference_mode():
        outputs = clip_model(pixel_values=batch["pixel_values"])
    # image_embeds are the projected features in CLIP's shared image-text space.
    return F.normalize(outputs.image_embeds.float(), dim=-1)


def encode_texts(texts):
    batch = clip_processor(text=texts, padding=True, truncation=True, return_tensors="pt")
    batch = {key: value.to(DEVICE) for key, value in batch.items()}
    with torch.inference_mode():
        outputs = clip_model(
            input_ids=batch["input_ids"],
            attention_mask=batch.get("attention_mask"),
        )
    # text_embeds are the projected features in the same shared space.
    return F.normalize(outputs.text_embeds.float(), dim=-1)


image_embedding = encode_images([kitten_image])
text_embedding = encode_texts(["An orange kitten lying on its back."])

print("Image embedding shape:", tuple(image_embedding.shape))
print("Text embedding shape: ", tuple(text_embedding.shape))
print("First 8 image values:  ", image_embedding[0, :8].cpu().numpy().round(3))
print("First 8 text values:   ", text_embedding[0, :8].cpu().numpy().round(3))

**Think about it (1 minute)**

1. What does one row of the image embedding represent?
2. Why are the image and text vectors the same length?
3. Can we interpret a single coordinate (for example, dimension 17) as “catness”?

<details>
<summary><strong>Expected interpretation</strong></summary>

Each row is a learned numerical representation of the complete input. CLIP maps both modalities into a common-dimensional space so that their similarity can be compared. Individual dimensions are not generally human-interpretable labels.
</details>

---

## 2. CLIP zero-shot classification (12–27 min)

We now compare one image against several candidate descriptions. This is **zero-shot classification**: we specify the candidate classes at inference time rather than training a new classifier for them.

### Predict before running

For the kitten image, which description do you expect to receive the highest similarity? What would happen if we changed the wording of a candidate?

In [ ]:
# STUDENT EDIT: change, remove or add candidate descriptions.
candidate_prompts = [
    "A photo of an orange kitten",
    "A photo of a golden retriever puppy",
    "A photo of a dog in a field of flowers",
    "A photo of a red car",
    "A photo of a bird",
]

candidate_embeddings = encode_texts(candidate_prompts)
zero_shot_scores = (image_embedding @ candidate_embeddings.T).squeeze(0).cpu().numpy()

ranking = np.argsort(zero_shot_scores)[::-1]
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh([candidate_prompts[i] for i in ranking][::-1], zero_shot_scores[ranking][::-1])
ax.set_xlabel("Cosine similarity")
ax.set_title("CLIP: image-to-text similarity")
plt.tight_layout()
plt.show()

print("Top match:", candidate_prompts[int(np.argmax(zero_shot_scores))])

### Experiment with the wording

Replace one candidate with a more detailed caption, for example:

- `A photo of a cat`
- `An orange kitten lying on its back with its paws raised`
- `A sleepy orange kitten on a white blanket`

Run the previous cell again. Compare the ranking.

**Questions**

1. Does a more detailed caption always receive a higher score?
2. Does the model compare the image with an unlimited set of possible classes?
3. Is the highest similarity a calibrated probability that the label is true?

<details>
<summary><strong>Expected interpretation</strong></summary>

CLIP ranks the candidate texts that we provide. Results can change with wording and with the candidate set. A cosine similarity is a compatibility score, not a universally calibrated probability. “Zero-shot” means that we did not train a new classifier for these candidate labels; it does not mean that the model has never encountered these concepts during pretraining.
</details>

---

## 3. Text-to-image retrieval (27–37 min)

Zero-shot classification compares one image with candidate texts. **Image retrieval reverses the search direction:** given a text query, rank a collection of images by similarity.

We encode the gallery once, then reuse the cached image embeddings for each query. A generative LLM is not needed for this ranking step.

In [ ]:
# Precompute one embedding per gallery image.
gallery_images = [item["image"] for item in gallery]
gallery_labels = [item["label"] for item in gallery]
gallery_embeddings = encode_images(gallery_images)
print("Gallery embedding matrix:", tuple(gallery_embeddings.shape))


def search_images(query, top_k=3):
    query_embedding = encode_texts([query])
    scores = (query_embedding @ gallery_embeddings.T).squeeze(0).cpu().numpy()
    order = np.argsort(scores)[::-1][:min(top_k, len(gallery))]

    fig, axes = plt.subplots(1, len(order), figsize=(4 * len(order), 3.5))
    if len(order) == 1:
        axes = [axes]
    for ax, idx in zip(axes, order):
        ax.imshow(gallery[idx]["image"])
        ax.set_title(f"{gallery_labels[idx]}\nscore = {scores[idx]:.3f}")
        ax.axis("off")
    plt.suptitle(f"Top matches for: {query}")
    plt.tight_layout()
    plt.show()

    return [(gallery_labels[idx], float(scores[idx])) for idx in order]


# STUDENT EDIT: change the text query.
text_query = "An orange kitten resting on its back."
search_images(text_query, top_k=3)

### Try a second query

Use a different description, for example `A golden retriever puppy playing in grass` or `A dog in a field of flowers`.

**Questions**

1. Does the system return the closest available image, even if none is a perfect match?
2. What would happen if the gallery contained 10,000 images instead of three?
3. Why can image embeddings be precomputed?

<details>
<summary><strong>Expected interpretation</strong></summary>

Retrieval ranks the images available in the gallery; the best result is not guaranteed to be a good match in an absolute sense. Image embeddings can be computed once and stored. For a large collection, an approximate nearest-neighbor index can make retrieval faster, but it is unnecessary for this small practical.
</details>

---

## 4. Visual question answering with a vision-language model (37–51 min)

CLIP gives us a similarity score between an image and candidate texts. It does not directly generate an open-ended answer to an arbitrary question.

Now we use an instruction-tuned vision-language model. It receives **both the image and the question**, then generates text. This is an example of multimodal fusion.

We use `HuggingFaceTB/SmolVLM2-2.2B-Instruct` as a compact demonstration model. You can replace it with a pre-downloaded **SmolVLM2-compatible** checkpoint by setting `VLM_MODEL_ID` in the configuration cell. Other model families may use a different chat-template or loading API and would require a small change to the inference helper. This model is used here for image understanding and text generation, not image generation.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText

vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)
vlm_model = AutoModelForImageTextToText.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype=MODEL_DTYPE,
).to(DEVICE).eval()

print("Vision-language model loaded.")

### Ask a question about the kitten

The function below prepares the image–text conversation, runs generation and returns only the newly generated answer. The model-specific chat template and image preprocessing are handled by the processor.

In [ ]:
def ask_vlm(image_path, question, max_new_tokens=100):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "path": str(image_path)},
                {"type": "text", "text": question},
            ],
        }
    ]

    inputs = vlm_processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(DEVICE, dtype=MODEL_DTYPE)

    with torch.inference_mode():
        output_ids = vlm_model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
        )

    prompt_length = inputs["input_ids"].shape[-1]
    answer_tokens = output_ids[0, prompt_length:]
    return vlm_processor.decode(answer_tokens, skip_special_tokens=True).strip()


opening_question = "Describe what you see in this image in one sentence."
opening_answer = ask_vlm(kitten_path, opening_question)

display(Markdown(f"**Question:** {opening_question}\n\n**Model answer:** {opening_answer}"))

### Probe the difference between observation and inference

Now ask a more careful question.

In [ ]:
# STUDENT EDIT: try your own question, or use the suggested one.
student_question = (
    "What observable details suggest that the kitten is resting? "
    "Can we be certain that it is asleep?"
)

student_answer = ask_vlm(kitten_path, student_question)
display(Markdown(f"**Question:** {student_question}\n\n**Model answer:** {student_answer}"))

**Compare the two models**

- CLIP compared the image with a fixed list of candidate descriptions.
- The vision-language model generated an answer conditioned on the image and the question.

Discuss: can a fluent answer still contain a false claim? Which details in the answer can you verify directly from the pixels?

> **Important:** A plausible response is not proof. The still image shows visible details, but it cannot establish an unseen event or the kitten's internal state with certainty.

---

## 5. How does visual information reach the LLM? (51–56 min)

The previous experiment showed a capability, not the model's complete internal architecture. The course introduced two public architectural examples that illustrate different fusion mechanisms.

### LLaVA-style representation injection

```text
Image → Vision encoder → Projection / connector ─┐
                                                  ├→ LLM input sequence → Text output
Question → Text token embeddings ────────────────┘
```

The projected visual representations are placed in the sequence processed by the language model, alongside the question tokens.

### Flamingo-style cross-attention

```text
Image / video → Vision encoder → Perceiver Resampler → Visual memory (K, V)
                                                        ↑
Question → Text states (Q) → Cross-attention ───────────┘
                               ↓
                         Text generation
```

The language model can consult a separate visual memory through added cross-attention layers. In this illustration, text states provide queries (`Q`), while visual memory provides keys and values (`K`, `V`).

| | LLaVA (original principle) | Flamingo |
|---|---|---|
| Vision encoder | Yes | Yes |
| Connector | Projection | Perceiver Resampler |
| Visual access | Visual positions in the LLM input sequence | Visual memory via cross-attention |
| Output in this course example | Text | Text |

**Discuss with a neighbour:** What is common to both approaches? What is the key difference in how the LLM receives visual information?

---

## 6. Final recap quiz (56–60 min)

Answer these without rerunning the models.

**1.** Which mechanism maps pixels or tokens to numerical vectors?

- A. Encoding
- B. Alignment
- C. Fusion

**2.** Why can CLIP perform text-to-image retrieval without a generative LLM?

- A. It generates captions for every image.
- B. It compares image and text embeddings in a shared space.
- C. It predicts bounding boxes.

**3.** What is the key distinction between the two architectures discussed?

- A. LLaVA uses no vision encoder, while Flamingo does.
- B. LLaVA injects projected visual representations into the LLM sequence; Flamingo uses cross-attention to visual memory.
- C. Flamingo generates images, while LLaVA generates text.

**4.** A model gives a fluent description of an image. Does that prove that every detail is correct? Explain in one sentence.

<details>
<summary><strong>Answers and explanations</strong></summary>

1. **A — Encoding.** Encoders convert input signals into representations.
2. **B — Shared-space comparison.** Retrieval can rank similarity scores without generating text.
3. **B — Different fusion mechanisms.** Both examples use a vision encoder, but their visual-to-language interfaces differ.
4. **No.** Generated answers can be plausible but incorrect; verify claims against the available visual evidence.
</details>

## Take-home message

The practical followed the same progression as the course:

- **Encoding** produces numerical representations of modalities.
- **Alignment** makes image and text representations comparable, enabling retrieval and zero-shot classification.
- **Fusion** lets a language model use visual information when generating a response.

A multimodal system's supported inputs and outputs do not, by themselves, reveal its exact internal architecture.

### Optional extension (only if you finish early)

Add more images to the local gallery and repeat the retrieval experiment with a more specific query. Then test whether changing only the wording changes the ranking.